# 01 — Setup: checkpoint check, data check, patient-level splits, baselines

**Session options:** Accelerator **None** (CPU) · Internet **On**

**Inputs to attach:** `jbhi-experiments` (scripts) · the memmap dataset shared by your teammate · `pretrain-v6-weights` (thesis weights) · `jbhi-pretrained-weights` (your laptop copy) · optional: your VitalDB clinical CSV (`clinical_data.csv` or `cases.csv`).

**How to use**
1. Run cells 1–4 interactively, then **stop at cell 5** and send Claude the printed output.
2. After deciding on the two settings in cell 5, run the rest, then **Save Version → Save & Run All (Commit)**. The commit's output (`splits/`, `runs/`) is what notebooks 02 and 04 attach.

## 1 — Find inputs and copy the scripts

In [ ]:
import glob, os, shutil, hashlib, json

def find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

# scripts
s = find('/kaggle/input/**/finetune.py')
assert s, 'Scripts not found: attach the jbhi-experiments dataset'
shutil.copytree(os.path.dirname(s[0]), '/kaggle/working/exp', dirs_exist_ok=True)
os.chdir('/kaggle/working/exp')
print('scripts  :', os.getcwd(), sorted(f for f in os.listdir('.') if f.endswith('.py')))

# memmaps = folder that holds train/offset_5m/n_samples.npy
mm = find('/kaggle/input/**/train/offset_5m/n_samples.npy')
assert mm, 'Memmaps not found: attach the shared physio_memmap dataset'
MEM_ALL = [os.path.dirname(os.path.dirname(os.path.dirname(p))) for p in mm]
MEM = MEM_ALL[0]
print('memmaps  :', MEM, '' if len(MEM_ALL) == 1 else f'(also found: {MEM_ALL[1:]})')

# encoder weights
ENCS  = find('/kaggle/input/**/hybrid_v4_encoders.pth')
PAIRS = find('/kaggle/input/**/ablation_*_encoders.pth')
FULLS = find('/kaggle/input/**/*_full.pth')
print('tri-modal encoders:', *ENCS, sep='\n   ')
print('pair encoders     :', *PAIRS, sep='\n   ')
print('full models       :', *FULLS, sep='\n   ')

# optional uploaded clinical table, otherwise the VitalDB API
CLIN_FILES = [p for p in find('/kaggle/input/**/*.csv') if os.path.basename(p).lower() in ('clinical_data.csv', 'cases.csv')]
CLIN = CLIN_FILES[0] if CLIN_FILES else 'https://api.vitaldb.net/cases'
print('clinical table    :', CLIN)

# anything else in the memmap dataset (e.g. old fold checkpoints) -- good to know about
root = os.path.dirname(MEM)
print('\nmemmap dataset top level:', root)
for p in sorted(os.listdir(root)):
    print('   ', p)

## 2 — What is in the encoder checkpoints? Are your two copies identical?

In [ ]:
def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

for p in ENCS:
    print(f'{md5(p)}  {os.path.getsize(p)/1024:,.0f} KB  {p}')
if len(ENCS) > 1:
    print('\nSAME FILE' if len({md5(p) for p in ENCS}) == 1 else '\n!! The copies DIFFER -- tell Claude which one the thesis used')

ARGS = ' '.join(f'"{p}"' for p in ENCS + PAIRS)
!python check_checkpoint.py {ARGS}

## 3 — Check the memmaps (every split and horizon)

In [ ]:
import numpy as np
W = ['offset_3m', 'offset_5m', 'offset_7m', 'offset_10m', 'offset_12m', 'offset_15m']
ok = True
print(f'{"horizon":11s} {"split":6s} {"N":>5s} {"pos":>4s} {"names":>6s} {"median ABP":>11s}')
for w in W:
    for sp in ['train', 'val', 'test']:
        d = os.path.join(MEM, sp, w)
        need = ['n_samples.npy', 'filenames.npy', 'ppg.dat', 'abp.dat', 'ecg.dat', 'labels.dat', 'scalars.dat',
                'map_targets.dat', 'sbp_targets.dat', 'dbp_targets.dat']
        miss = [f for f in need if not os.path.exists(os.path.join(d, f))]
        if miss:
            ok = False
            print(f'{w:11s} {sp:6s} MISSING {miss}')
            continue
        N = int(np.load(os.path.join(d, 'n_samples.npy'))[0])
        fn = np.load(os.path.join(d, 'filenames.npy'), allow_pickle=True)
        lab = np.memmap(os.path.join(d, 'labels.dat'), dtype='int32', mode='r', shape=(N,))
        abp = np.memmap(os.path.join(d, 'abp.dat'), dtype='float32', mode='r', shape=(N, 3750))
        med = float(np.median(abp[:min(N, 200)]))
        ok &= len(fn) == N
        print(f'{w:11s} {sp:6s} {N:5d} {int(lab.sum()):4d} {len(fn):6d} {med:11.1f}')
print('\nALL OK' if ok else '\n!! Problems above -- send this output to Claude')
print('Median ABP should look like mmHg (roughly 60-110). Example file name:', fn[0])

## 4 — Patient-level splits, leakage check, Table 1 (first pass)

In [ ]:
!python make_splits.py --memmap_dir "{MEM}" --vitaldb_cases "{CLIN}" --out_dir /kaggle/working/splits_firstpass

## 5 — STOP: send Claude the output of cells 1–4

Two decisions depend on it. Set them below, then run the remaining cells:

* **`DROP_BOLUS_CONTROLS`** — "control" cases that got an intra-op ephedrine / phenylephrine / epinephrine bolus (listed by cell 4). `True` removes them everywhere, *including the test set*, so every number changes. Decide once, before any GPU run.
* **`KEEP_LEAKS`** — keep `False` (train/val windows of patients who are also in test are excluded; the test set is unchanged).

In [ ]:
DROP_BOLUS_CONTROLS = False   # <- set after talking to Claude
KEEP_LEAKS = False

## 6 — Final splits (this is what every later notebook uses)

In [ ]:
extra = ''
bolus = '/kaggle/working/splits_firstpass/controls_with_bolus.csv'
if DROP_BOLUS_CONTROLS and os.path.exists(bolus):
    extra += f' --drop_cases {bolus}'
if KEEP_LEAKS:
    extra += ' --keep_leaks'
!python make_splits.py --memmap_dir "{MEM}" --vitaldb_cases "{CLIN}" --out_dir /kaggle/working/splits {extra}
json.dump(dict(MEM=MEM, CLIN=CLIN, ENCS=ENCS, PAIRS=PAIRS, DROP_BOLUS_CONTROLS=DROP_BOLUS_CONTROLS,
               KEEP_LEAKS=KEEP_LEAKS), open('/kaggle/working/splits/setup_info.json', 'w'), indent=2)
shutil.rmtree('/kaggle/working/splits_firstpass', ignore_errors=True)

## 7 — Naive baselines and persistence forecast (CPU, a few minutes)

In [ ]:
!python baselines_tabular.py --memmap_dir "{MEM}" --windows_csv /kaggle/working/splits/windows.csv --out_dir /kaggle/working/runs

## 8 — Quick look at the baseline results

In [ ]:
!python evaluate.py --runs_dir /kaggle/working/runs --reference baseline_lastmap --n_boot 500
import pandas as pd
r = pd.read_csv('/kaggle/working/runs/_eval/results_long.csv')
cols = [c for c in ['run', 'window', 'n', 'n_pos', 'n_patients', 'auroc', 'auroc_lo', 'auroc_hi', 'auprc', 'mae_map', 'mae_sbp', 'mae_dbp'] if c in r]
r[cols].round(3)

## 9 — Save

**Save Version → Save & Run All (Commit)**. When it finishes, the version's **Output** contains `splits/windows.csv`, `splits/report.txt`, `splits/cohort_table.csv` and `runs/baseline_*`. Notebook 02 attaches this output.

Clean-up so only results are saved:

In [ ]:
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/exp', ignore_errors=True)   # scripts are re-copied from the dataset each time
!find /kaggle/working -maxdepth 2 | sort